# Guided Solver Config Builder — Step 1 of 2

This notebook configures all model inputs and writes a `solver_params.json` file to disk. **Run this before `guided_bayesian_inference.ipynb`.**

The four sections below cover: output file names, free parameters, experimental datasets (including the calculation module), and solver settings. Most users only need to edit **Sections 1, 2, and 3**.

*Annette Thompson · Fox and Shirts Labs, CU Boulder · 2026 · Developed with assistance from GitHub Copilot (Claude Sonnet 4.6)*

### What is `solver_params.json`?

It is a single JSON file that bundles everything the inference engine needs:
- which kinetic parameters to infer (and their prior distributions)
- which experimental datasets to fit
- ODE solver tolerances and MCMC sampler settings
- per-dataset initial condition definitions (from `init_cond_columns` and/or `init_cond_overrides`)

Run the cells below in order, then execute the final cell to write the file to disk.

In [1]:
import sys
sys.path.append("../")
from Utilities.solver_config_builder import build_solver_params_config, write_solver_params_json_file

## Section 1: Output file names

In [2]:
# Change `folder_name` to choose where files are read/written.
enzyme_name = "FabD"
folder_name = "FabD scaling inference"

# This notebook writes the solver JSON from the notebook working directory.
solver_params_file = f"solver_params.json"
results_save_dir = f"../Results/{folder_name}"
solver_file_dir = results_save_dir

# `path_base` is the model home directory, stored relative to the solver JSON file.
# All other paths inside solver_params.json are relative to this home directory.
path_base = "../.."

output_paths = {
    "reactions_source": f"Reactions/EC_FAS_ME1/{enzyme_name}.yaml",
    "results_save_dir": f"Results/{folder_name}",
    "prior_samples_file": "prior_samples_pm.nc",
    "posterior_samples_file": "posterior_samples_pm.nc",
    "trace_plot_file": "trace_plot.png",
}

## Section 2: Free Parameters & Priors

List every model parameter you want to infer. This can be a reaction rate constant or a scaling parameter such as `a1` used in reaction-file `scaling_group` / `rvs_scaling_group` expressions.

| Key | Description |
|---|---|
| `rxn_name` | Optional reaction name for documentation; scaling parameters can leave this out |
| `param_name` | Name of the parameter in the reaction-model parameter vector, e.g. a rate constant or `a1` |
| `distribution` | Prior shape — `"Gamma"`, `"LogNormal"`, `"Normal"` (any `preliz.maxent` distribution option)|
| `lower` / `upper` | Interval bounds for the prior |
| `mass` | Probability mass within `[lower, upper]` - % of mass in distribution that should fall between designated bounds |

The prior is constructed via **maximum entropy** (`preliz.maxent`) — the least-informative (most uncertain) distribution consistent with your bounds. If `param_name` is a scaling parameter like `a1`, every reaction rate expression that uses `a1` is recalculated from the sampled value during inference.

In [3]:
# Each entry specifies one model parameter to infer.
# Here we infer FabD scaling parameters only.

free_parameter_prior_inputs = [
    {"param_name": "a1", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    {"param_name": "b1", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    {"param_name": "b2", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    {"param_name": "b3", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
]

## Section 3: Experimental Datasets + Calculation Module

Each entry points to a CSV file and tells the model how to interpret it.

Also set `calculation_module_path` in this section. This is the Python file that defines `OBSERVABLES` (for FabD: `FabD_calculations.py`).

**Current supported dataset types:**
- `"endpoint"` — a single measurement at a fixed time (e.g., final product at t = 150 s). Set `time_values` to the measurement time(s).
- `"timeseries"` — measurements at multiple time points. Set `time_column` to the name of the time column in the CSV.


If you want to add error into the data (mock experiment/model error during testing), you can use a noise model.

**Noise models:**
- `"relative_mean"` - sets observation noise to % (parameter = `"frac"`) of the mean absolute signal
- `"relative_pointwise"`  - sets observation noise to % (parameter = `"frac"`) of their respective values
- `"absolute"` - sets observation noise to a constat sigma value (parameter = `"value"`)
- `"relative_plus_floor"` - same as relative_pointwise but adds constant value to each sigma (parameter = `"floor`")
- `"column"` - set observation noise to sigma column in dataset (parameter = `"column_mapping"`)
- `"groupwise"` - sets observation noise based on a group-specific variability statistic (parameter = `"statistic"`, options: "std", "sem", "mad") calculated across subsets defined by a matching column (parameter = `"group_column"`)

**`init_cond_columns`** (endpoint datasets only) — maps species names to CSV column names that hold per-row initial conditions (e.g., varying enzyme concentrations across different wells).

**`observables`** - specifies one or more calculated outputs from your simulation module to be compared against that dataset and maps them to the column in the experiment file (observable name in calculation file: observable name in experiment csv)

Initial conditions must come from each dataset description (`init_cond_columns` and/or `init_cond_overrides`); there are no global defaults in the config. Each dataset must define at least one initial condition source.



In [4]:
# Path to the calculation module that defines OBSERVABLES for simulation outputs.
calculation_module_path = f"Calculation Files/{enzyme_name}/FabD_calculations.py"

# List every CSV dataset to include in the inference.
dataset_inputs = [
    {
        "name": "conc_vs_final_C3_MalACP",
        "dataset_type": "endpoint",
        "data_file": f"Data/{enzyme_name}/conc_vs_final_C3_MalACP.csv",
        "observables": {
            "MalACP (uM)": "C3_MalACP (uM)",
            },
        "time_values": [150],
        "init_cond_columns": {
            "FabD": "FabD (uM)",
            "C3_MalCoA": "C3_MalCoA (uM)",
            "ACP": "ACP (uM)",
        },
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.05},
    },
    {
        "name": "time_vs_C3_MalACP",
        "dataset_type": "timeseries",
        "data_file": f"Data/{enzyme_name}/time_vs_C3_MalACP.csv",
        "observables": {
            "MalACP (uM)": "C3_MalACP (uM)",
            },
        "time_column": "Time (s)",
        # Provide fixed ICs for species not present as columns in this dataset.
        "init_cond_overrides": {
            "FabD": 0.001,
            "C3_MalCoA": 0.1,
            "ACP": 10,
        },
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.05},
    },
]

## Section 4: Sampling & Solver Settings

Controls how many MCMC samples are drawn and the ODE integrator tolerances.

| Setting | Notes |
|---|---|
| `draws` (posterior) | Samples per chain after tuning |
| `tune` | Warm-up steps per chain (discarded) |
| `chains` | Parallel chains |
| `target_accept` | NUTS acceptance rate; `0.8` is the current fast stable default |
| `nuts_sampler` | `"numpyro"` uses the vectorized JAX sampling path used by this workflow |
| `rtol` / `atol` | ODE solver tolerances; keep these at the stable defaults unless debugging |

> **Leave `ode_solver_settings` and `ode_stepsize_controller_settings` unchanged** unless you are debugging stiff ODE issues. Loosening these tolerances can make gradients fail even if the forward solve appears faster.

In [5]:
# Controls MCMC sampling and the ODE integrator.

# Using fewer draws and looser ODE solver settings for debugging; increase for production runs.
prior_sampling_settings = {"draws": 10000, "random_seed": 0}
posterior_sampling_settings = {
    "draws": 100,
    "tune": 400,
    "nuts_sampler": "nutpie",
    "target_accept": 0.95,
}

ode_solver_settings = {
    "solver_name": "Kvaerno5",
    "max_steps": 100_000,
    "dt0": 1e-6,
    "stepsize_controller": "PIDController",
    "robust_linear_solver": True,
}

ode_stepsize_controller_settings = {
    "rtol": 1.0e-5,
    "atol": 1.0e-8,
    "pcoeff": 0.2,
    "icoeff": 0.4,
    "dcoeff": 0.0,
}

---
## Build and Write the Config File

Run this cell to assemble all settings into a single `solver_params.json` file and write it to disk. The output path will be printed when done.

After this cell succeeds, open **`guided_bayesian_inference.ipynb`** and make sure the `name` variable there matches the `name` set in Section 1 above.

In [6]:
solver_params = build_solver_params_config(
    free_parameter_prior_inputs=free_parameter_prior_inputs,
    dataset_inputs=dataset_inputs,
    prior_sampling_settings=prior_sampling_settings,
    posterior_sampling_settings=posterior_sampling_settings,
    ode_solver_settings=ode_solver_settings,
    ode_stepsize_controller_settings=ode_stepsize_controller_settings,
    calculation_module_path=calculation_module_path,
    path_base=path_base,
    output_paths=output_paths,
)

written_solver_params_path = write_solver_params_json_file(
    solver_params_config=solver_params,
    file_directory=solver_file_dir,
    filename=solver_params_file,
)

print(f"Wrote solver config: {written_solver_params_path}")
print(f"Folder name: {folder_name}")
print(f"Path base: {path_base}")
print(f"Reaction source: {output_paths['reactions_source']}")
print(f"Save directory: {output_paths['results_save_dir']}")
print(
    f"Configured free params: {[param['param_name'] for param in solver_params['free_kinetic_params']]}" 
)
print(
    f"Configured datasets: {[dataset['name'] for dataset in solver_params['datasets']]}" 
)

Wrote solver config: /Users/annettethompson/Library/CloudStorage/OneDrive-SharedLibraries-UCB-O365/Jerome Michael Fox - Annie Thompson/Git Repositories/Bayesian Kinetic Model/Python Model/Results/FabD scaling inference/solver_params.json
Folder name: FabD scaling inference
Path base: ../..
Reaction source: Reactions/EC_FAS_ME1/FabD.yaml
Save directory: Results/FabD scaling inference
Configured free params: ['a1', 'b1', 'b2', 'b3']
Configured datasets: ['conc_vs_final_C3_MalACP', 'time_vs_C3_MalACP']
